# 05 - Create & publish the Fabric Data Agent

Run this **inside Fabric** (it uses the `fabric-data-agent-sdk`, which relies on the
Fabric runtime for auth and .NET). It creates/updates the Telco data agent, attaches
the Lakehouse, sets instructions + example queries, and publishes.

**Steps:** attach the Telco Lakehouse as the default Lakehouse, then **Run all**. Copy
the printed `DATA_AGENT_ARTIFACT_ID` and `DATA_AGENT_MCP_ENDPOINT` into your local `.env`.

> This is the supported way to create the data agent (it is not created from the local
> workstation). Config below mirrors `fabric/data-agent/config.yaml`, the source of truth.

In [ ]:
%pip install --quiet fabric-data-agent-sdk

## Configuration (mirrors fabric/data-agent/config.yaml)

In [ ]:
NAME = 'TelcoCustomerServiceAgent'
DESCRIPTION = 'Answers telecommunications customer-service questions over the Telco Lakehouse: billing and first-bill support, cross-sell/upsell, service health, outages, and retention/churn risk.\n'
LAKEHOUSE_DEFAULT = 'TelcoLakehouse'  # fallback only
GOLD_SCHEMA = 'gold'
GOLD_TABLES = ['dim_geography', 'dim_product', 'dim_plan', 'dim_device', 'dim_promotion', 'dim_customer', 'dim_account', 'fact_subscription', 'fact_invoice', 'fact_invoice_line', 'fact_usage_data', 'fact_usage_voice', 'fact_coverage', 'fact_outage', 'fact_service_metric', 'fact_work_order', 'fact_appointment', 'fact_contact', 'fact_offer', 'fact_feedback', 'ml_churn_score', 'ml_crosssell_reco', 'customer_360']
AI_INSTRUCTIONS = 'You are a data agent for a telecommunications customer-service team. You answer\nquestions about customers, accounts, billing, subscriptions, usage, service health,\noutages, work orders, offers, and churn/cross-sell scores.\n\nThe data is in the Lakehouse "gold" schema. Reference tables as gold.<table>.\n\nGuidance:\n- Prefer the curated gold tables (gold.dim_*, gold.fact_*, gold.ml_*, gold.customer_360).\n- gold.customer_360 is a denormalized per-customer profile; use it for "give me a 360 view".\n- "First bill" means gold.fact_invoice.is_first_bill = true. A confusing/high first bill\n  usually includes one-time activation and proration lines (gold.fact_invoice_line).\n- "At-risk" / "churn risk" customers are in gold.ml_churn_score (risk_band High/Medium/Low).\n- Cross-sell candidates and recommendations are in gold.ml_crosssell_reco.\n- "Outage" or "service degradation" -> gold.fact_outage (by geography) and\n  gold.fact_service_metric (per-account latency/packet loss/uptime).\n- Always scope monetary answers to the correct account and time period.\n- Never invent values; only answer from the data.\n'
DS_INSTRUCTIONS = 'This Lakehouse uses a medallion layout with schemas: bronze (raw), silver (empty for now),\nand gold (curated). Query the gold-schema tables (gold.dim_*, gold.fact_*, gold.ml_*,\ngold.customer_360). Join keys: customer_id, account_id, geo_id, product_id, plan_id,\ninvoice_id, work_order_id.\n'
EXAMPLE_QUERIES = [('Show me the first bill details for account ACCT000123', "SELECT il.description, il.category, il.amount\nFROM gold.fact_invoice i\nJOIN gold.fact_invoice_line il ON i.invoice_id = il.invoice_id\nWHERE i.account_id = 'ACCT000123' AND i.is_first_bill = true\nORDER BY il.amount DESC\n"), ('Which new customers have an unpaid first bill?', 'SELECT customer_id, first_name, last_name, last_invoice_amount, open_balance\nFROM gold.customer_360\nWHERE last_invoice_is_first_bill = true AND last_invoice_paid = false\n'), ('Which internet customers should be offered mobile?', "SELECT r.account_id, r.score, r.rationale\nFROM gold.ml_crosssell_reco r\nWHERE r.recommended_product_id = 'PROD_MOB'\nORDER BY r.score DESC\n"), ('What products does customer CUST000045 already have?', "SELECT s.product_id, s.plan_id, s.mrc, s.status\nFROM gold.fact_subscription s\nJOIN gold.dim_account a ON s.account_id = a.account_id\nWHERE a.customer_id = 'CUST000045'\n"), ('Which active customers are high churn risk and had a recent outage?', "SELECT customer_id, first_name, last_name, churn_probability, churn_top_reason\nFROM gold.customer_360\nWHERE risk_band = 'High' AND account_status = 'active'\n  AND recent_outage_exposure = true\nORDER BY churn_probability DESC\n"), ('What is the average uptime for account ACCT000123 over the last two weeks?', "SELECT ROUND(AVG(uptime_pct), 2) AS avg_uptime,\n       ROUND(AVG(latency_ms), 1) AS avg_latency_ms\nFROM gold.fact_service_metric\nWHERE account_id = 'ACCT000123'\n")]

## Resolve the current workspace + attached Lakehouse

The Lakehouse is taken from whichever Lakehouse you attach as the notebook's default
(so it works no matter what you named it, e.g. `lh_telco`). `LAKEHOUSE_DEFAULT` is only
used as a fallback if none is attached.

In [ ]:
workspace_id = None
LAKEHOUSE = None
try:
    import notebookutils
    ctx = notebookutils.runtime.context
    workspace_id = ctx.get('currentWorkspaceId')
    LAKEHOUSE = ctx.get('defaultLakehouseName')
except Exception:
    pass
if not workspace_id:
    import sempy.fabric as fabric
    workspace_id = fabric.get_notebook_workspace_id()
if not LAKEHOUSE:
    LAKEHOUSE = LAKEHOUSE_DEFAULT
    print(f'WARNING: no default Lakehouse detected - falling back to {LAKEHOUSE!r}.')
    print('         Attach your Lakehouse (Explorer panel) as the default and re-run',
          'if this is not the one holding the loaded tables.')
print('Workspace:', workspace_id)
print('Lakehouse:', LAKEHOUSE)

## Create, configure, and publish

Note: `add_staging_datasource()` returns a raw `Response`; the configurable
`Datasource` object (with `select` / `add_fewshots` / `update_configuration`) comes
from `agent.get_datasources()`. The tree is printed before/after so you can see the
exact table paths and confirm what got selected.

In [ ]:
from fabric.dataagent.client import create_data_agent

agent = create_data_agent(data_agent_name=NAME, workspace_id=workspace_id)
agent.update_settings(ai_instructions=AI_INSTRUCTIONS)
print('Applied AI instructions.')

# add_staging_datasource returns a Response; fetch the configurable Datasource object.
agent.add_staging_datasource(
    artifact_name_or_id=LAKEHOUSE, workspace_id_or_name=workspace_id)
datasources = agent.get_datasources()
if not datasources:
    raise RuntimeError('No datasource found after add_staging_datasource; '
                       'check that the attached Lakehouse name is correct.')
ds = datasources[-1]
print(f'Configured datasource for {LAKEHOUSE} ({len(datasources)} total).')

In [ ]:
# Show the available element tree so you can see valid table paths.
print('--- datasource tree (before select) ---')
try:
    ds.pretty_print()
except Exception as _ex:
    print('pretty_print unavailable:', _ex)

In [ ]:
def _try_select(*path):
    try:
        ds.select(*path)
        return True
    except Exception:
        return False

# Prefer selecting the whole gold schema; fall back to per-table paths.
selected = False
if _try_select(GOLD_SCHEMA):
    print(f'Selected the entire {GOLD_SCHEMA} schema.')
    selected = True
else:
    n = 0
    for _t in GOLD_TABLES:
        if _try_select(GOLD_SCHEMA, _t) or _try_select(_t):
            n += 1
    if n:
        print(f'Selected {n}/{len(GOLD_TABLES)} gold tables.')
        selected = True
if not selected:
    print('Could not auto-select tables. Use the tree above to pick the correct path,')
    print('e.g. ds.select("<schema>", "<table>"), or select them in the Data Agent UI.')

In [ ]:
# Datasource instructions.
if DS_INSTRUCTIONS:
    try:
        ds.update_configuration(instructions=DS_INSTRUCTIONS)
        print('Applied datasource instructions.')
    except Exception as _ex:
        print('update_configuration failed:', _ex)

# Example queries (few-shots): a dict of {question: query}.
if EXAMPLE_QUERIES:
    try:
        ds.add_fewshots({q: s for q, s in EXAMPLE_QUERIES})
        print(f'Added {len(EXAMPLE_QUERIES)} example queries (few-shots).')
    except Exception as _ex:
        print('add_fewshots failed:', _ex)

print('--- datasource tree (after select) ---')
try:
    ds.pretty_print()
except Exception:
    pass

In [ ]:
agent.publish_staging(description=DESCRIPTION)
print('Published data agent.')

## Data agent id + MCP endpoint

In [ ]:
aid = None
for _a in ('id', 'artifact_id', 'data_agent_id'):
    aid = getattr(agent, _a, None)
    if aid:
        break
if aid:
    print('DATA_AGENT_ARTIFACT_ID =', aid)
    print('DATA_AGENT_MCP_ENDPOINT =',
          f'https://api.fabric.microsoft.com/v1/mcp/workspaces/{workspace_id}/dataagents/{aid}/agent')
    print('\nAdd these to your local .env so the Foundry agents can bind to the data agent.')
else:
    print('Published. Copy the data agent id + MCP URL from the agent settings ->',
          'Model Context Protocol tab into .env.')